In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

In [2]:
df = pd.read_csv('../../data/features/mordred_combined_df.csv')

/tmp/ipykernel_56150/3779524202.py:1: DtypeWarning: Columns (875,883,1097,1099,1102,1107,1176,1178,1181,1186,1569,1570,1571) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('mordred_combined_df.csv')


In [3]:
df.columns

Index(['Unnamed: 0', 'ABC', 'ABCGG', 'nAcid', 'nBase', 'SpAbs_A', 'SpMax_A',
       'SpDiam_A', 'SpAD_A', 'SpMAD_A',
       ...
       'SRW10', 'TSRW10', 'MW', 'AMW', 'WPath', 'WPol', 'Zagreb1', 'Zagreb2',
       'mZagreb1', 'mZagreb2'],
      dtype='object', length=1827)

In [4]:
columns_to_drop = df.columns[:1].tolist()
X = df.drop(columns = columns_to_drop, axis=1)

In [5]:
X.shape

(1481, 1826)

In [6]:
target_df = pd.read_csv('../../data/processed/Combined_df.csv')

In [7]:
target_df.columns

Index(['Unnamed: 0', 'SMILES', 'PCE'], dtype='object')

In [8]:
y = target_df['PCE']

In [9]:
y.shape

(1481,)

In [10]:
X.fillna(0, inplace = True)
X=X.select_dtypes(include='number')
X = X.astype(np.float32)
max_value = np.finfo(np.float32).max
X = X.loc[:, (X.iloc[:,:] <= max_value).all(axis=0)]

In [11]:
X.shape

(1481, 1118)

In [12]:
def remove_low_variance(input_data, threshold=0.1):
    selection = VarianceThreshold(threshold)
    selection.fit(input_data)
    return input_data[input_data.columns[selection.get_support(indices=True)]]

In [13]:
X = remove_low_variance(X, threshold=0.1)
X

,nAromAtom,nAromBond,nAtom,nHeavyAtom,nBridgehead,nHetero,nH,nC,nN,nO,...,SRW09,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb2
0,64.0,64.0,456.0,226.0,0.0,32.0,230.0,194.0,16.0,8.0,...,9.736311,12.286781,293.989044,3167.584717,6.946458,513101.0,376.0,1158.0,1381.0,52.555557
1,28.0,28.0,152.0,80.0,0.0,8.0,72.0,72.0,0.0,4.0,...,8.986321,11.562639,140.978165,1102.484009,7.253184,30062.0,155.0,432.0,538.0,18.444445
2,58.0,60.0,196.0,110.0,0.0,10.0,86.0,100.0,0.0,2.0,...,9.246576,11.941384,173.610703,1522.544678,7.768085,69795.0,219.0,616.0,772.0,24.444445
3,58.0,60.0,198.0,112.0,0.0,12.0,86.0,100.0,0.0,4.0,...,9.246576,11.953197,175.672134,1554.534546,7.851184,71896.0,223.0,624.0,782.0,25.111111
4,20.0,20.0,123.0,61.0,0.0,8.0,62.0,53.0,4.0,0.0,...,8.594339,10.963203,118.906281,882.385742,7.173868,15940.0,97.0,306.0,363.0,14.527778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1476,39.0,46.0,211.0,109.0,0.0,19.0,102.0,90.0,8.0,2.0,...,9.300638,11.726431,172.258957,1626.548340,7.708760,71453.0,194.0,578.0,706.0,25.000000
1477,68.0,70.0,250.0,128.0,0.0,18.0,122.0,110.0,12.0,2.0,...,9.151970,11.921087,191.751358,1770.869629,7.083478,109522.0,233.0,684.0,836.0,29.277779
1478,68.0,70.0,200.0,112.0,0.0,18.0,88.0,94.0,10.0,2.0,...,9.109082,11.871250,175.267578,1580.541626,7.902708,77706.0,213.0,616.0,764.0,25.277779
1479,31.0,33.0,220.0,103.0,0.0,15.0,117.0,88.0,5.0,4.0,...,8.851521,11.508245,164.494736,1499.743042,6.817014,69304.0,170.0,524.0,624.0,23.972221


In [14]:
X.shape

(1481, 577)

In [15]:
X.columns

Index(['nAromAtom', 'nAromBond', 'nAtom', 'nHeavyAtom', 'nBridgehead',
       'nHetero', 'nH', 'nC', 'nN', 'nO',
       ...
       'SRW09', 'SRW10', 'TSRW10', 'MW', 'AMW', 'WPath', 'WPol', 'Zagreb1',
       'Zagreb2', 'mZagreb2'],
      dtype='object', length=577)

In [16]:
def correlation(dataset, threshold):
    col_corr = set()  # Set of all the names of correlated columns
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if (corr_matrix.iloc[i, j]) > threshold: # we are interested in absolute coeff value
                colname = corr_matrix.columns[i]  # getting the name of column
                col_corr.add(colname)
    return col_corr
corr_features = correlation(X, 0.8)
print('Features with correlation more than 0.8 :',len(set(corr_features)))
X = X.drop(list(corr_features & set(X.columns)), axis=1)

Features with correlation more than 0.8 : 470


In [17]:
X.shape

(1481, 107)

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
X_train.shape, X_test.shape

((1184, 107), (297, 107))

In [20]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rf = RandomForestRegressor(random_state=42)

param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)], # Number of trees in the forest
    'max_depth': [int(x) for x in np.linspace(10, 110, num = 11)], # Maximum depth of the tree
    'min_samples_split': [2, 5, 10],                   # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],                     # Minimum number of samples required to be at a leaf node
    'max_features': ['auto', 'sqrt', 'log2'],           # Number of features to consider at every split
    'bootstrap': [True, False]                         # Whether to use bootstrap samples when building trees
}

random_search = RandomizedSearchCV(estimator=rf, 
                                   param_distributions=param_dist, 
                                   n_iter=100,        # Number of iterations for searching the parameter space
                                   scoring='r2',  # Metric to optimize
                                   cv=5,              # 5-fold cross-validation
                                   verbose=2,         # Show progress
                                   random_state=42, 
                                   n_jobs=-1)         # Use all available cores

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)

y_pred = random_search.best_estimator_.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'R² Score: {r2}')

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END bootstrap=False, max_depth=70, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=600; total time=   6.3s
[CV] END bootstrap=False, max_depth=70, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=600; total time=   6.0s
[CV] END bootstrap=False, max_depth=70, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=400; total time=   3.3s
[CV] END bootstrap=False, max_depth=70, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=400; total time=   3.4s
[CV] END bootstrap=True, max_depth=90, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=1800; total time=  11.1s
[CV] END bootstrap=False, max_depth=20, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=1400; total time=  11.8s
[CV] END bootstrap=True, max_depth=80, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=1

/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
120 fits failed out of a total of 500.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
105 fits failed with the following error:
Traceback (most recent call last):
  File "/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/base.py", line 1466, in wrapper
    estimator._validate_params()
  File "/home/hamza/anaconda3/envs/active_learning/lib/python3.12/site-packages/sklearn/base.py"

Best Parameters: {'n_estimators': 1800, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 100, 'bootstrap': False}
Mean Squared Error: 5.08436852044187
R² Score: 0.7097640564405734

[CV] END bootstrap=False, max_depth=10, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=1600; total time=   9.6s
[CV] END bootstrap=False, max_depth=90, max_features=log2, min_samples_leaf=4, min_samples_split=2, n_estimators=800; total time=   5.2s
[CV] END bootstrap=True, max_depth=60, max_features=log2, min_samples_leaf=2, min_samples_split=2, n_estimators=600; total time=   3.4s
[CV] END bootstrap=True, max_depth=90, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=400; total time=   3.9s
[CV] END bootstrap=True, max_depth=90, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=1800; total time=  11.8s
[CV] END bootstrap=True, max_depth=60, max_features=log2, min_samples_leaf=1, min_samples_split=2, 